In [1]:
import os
import json
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from string import Template
import time

In [5]:
print("👍data is loading ")
try:
    file_path = r"C:\Users\divya\OneDrive\Desktop\LAB SECURITY.pdf"
    pdf_loader = PyPDFLoader(file_path)
    docs = pdf_loader.load()
    print("✅ pdf loaded successfully")
except Exception as e:
    print(f"❌ error during reading the file - {e}")   

Ignoring wrong pointing object 2083 0 (offset 0)


👍data is loading 
✅ pdf loaded successfully


In [6]:
# metadata - data about data
print(f"number of pages in the pdf - {len(docs)}")

number of pages in the pdf - 58


In [7]:
print(type(docs[0]))

<class 'langchain_core.documents.base.Document'>


In [9]:
all_chunks = []

token_text_splitter = TokenTextSplitter(chunk_size = 100, chunk_overlap = 30)

print("🚀 Chunking started ...")
time.sleep(1.5)
try:
    for doc in docs:
        chunks = token_text_splitter.split_text(doc.page_content)
        all_chunks.extend(chunks)
    print("✅ Chunking successfully completed")
    print(f"Number of chunks extracted - {len(all_chunks)}")
except Exception as e:
    print(f"❌ Error during chunking - {e}")

🚀 Chunking started ...
✅ Chunking successfully completed
Number of chunks extracted - 431


In [10]:

try:
    print("🚀 Loading the Embedding model")
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("✅ Succesfully Initialized the embedding model")
except Exception as e:
    print(f"Error during Intialiazing the embedding model - {e}")
    

🚀 Loading the Embedding model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Succesfully Initialized the embedding model


In [11]:
embeddings = []

try:
    print("🚀 Converting chunks to embeddings ...")
    print()
    for i, chunk_text in enumerate(all_chunks):
        try:
            embedding = embedding_model.encode(chunk_text)
            embeddings.append(embedding)
            print(f"✅ {i+1} chunk succesfully converted into embedding")
        except Exception as e:
            print(f"❌ Error during embedding conversion - {e}")
    embeddings =  np.array(embeddings)
    print()
    print("✅ all the chunks have been converted into embeddings")
except Exception as e:
    print(f"Error during converting all the chunks to embeddings - {e}")

🚀 Converting chunks to embeddings ...

✅ 1 chunk succesfully converted into embedding
✅ 2 chunk succesfully converted into embedding
✅ 3 chunk succesfully converted into embedding
✅ 4 chunk succesfully converted into embedding
✅ 5 chunk succesfully converted into embedding
✅ 6 chunk succesfully converted into embedding
✅ 7 chunk succesfully converted into embedding
✅ 8 chunk succesfully converted into embedding
✅ 9 chunk succesfully converted into embedding
✅ 10 chunk succesfully converted into embedding
✅ 11 chunk succesfully converted into embedding
✅ 12 chunk succesfully converted into embedding
✅ 13 chunk succesfully converted into embedding
✅ 14 chunk succesfully converted into embedding
✅ 15 chunk succesfully converted into embedding
✅ 16 chunk succesfully converted into embedding
✅ 17 chunk succesfully converted into embedding
✅ 18 chunk succesfully converted into embedding
✅ 19 chunk succesfully converted into embedding
✅ 20 chunk succesfully converted into embedding
✅ 21 chunk

In [12]:
print(type(embeddings))

<class 'numpy.ndarray'>


In [13]:
print(f"shape of embeddings - {embeddings[0].shape}")

dim = embeddings[0].shape[0]
print(dim)

shape of embeddings - (384,)
384


In [14]:
try:
    print("🚀 Building the Index ...")
    index = faiss.IndexFlatL2(dim)
    if index.is_trained:
        print("✅ Successfully built the Index")
except Exception as e:
    print(f"❌ Error during building the Index - {e}")

🚀 Building the Index ...
✅ Successfully built the Index


In [15]:
try:
    print("Adding the embeddings into the index ...")
    index.add(embeddings)
    print("✅ Sucessfully added the embeddings into the Index")
except Exception as e:
    print(f"❌ Error during adding the Embeddings into the Index - {e}")

Adding the embeddings into the index ...
✅ Sucessfully added the embeddings into the Index


In [16]:
print(f"number of embeddings in the Index - {index.ntotal}")

number of embeddings in the Index - 431


In [17]:
question = input("💬 Ask your question: ")

query_embedding = embedding_model.encode([question])
print(query_embedding.shape) # dim should align with the chunk embeddings

(1, 384)


In [18]:
k = 2

D, I = index.search(query_embedding, k)
top_index = I.tolist()[0]
top_index = set(top_index)
top_index

{297, 301}

In [19]:
top_chunks = []
for i, chunk in enumerate(all_chunks):
    if i in top_index:
        top_chunks.append(chunk)
top_chunks

['\n\uf0a2 After the scan completes, the N -Stalker Report Manager will prompt \uf0a2 you to \nselect a format for the resulting report as choose Generate HTML. \uf0a2 Review the \nHTML report for vulnerabilities.  \n  \nEXPLORING N-STALKER:  \n  \n• N-Stalker Web Application Security Scanner is a Web security assessment tool.  \n• It incorporates with a well-known',
 ' the N-Stalker Report Manager will prompt 5. you to \nselect a format for the resulting report as choose Generate HTML.  \n6. Review the HTML report for vulnerabilities.']

In [20]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)
    return chunks

In [21]:
load_dotenv() # loads all the env variables into our file

try:
    print("🚀 Loading API KEY...")
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("API key shouldn't be empty or None")
    print("✅ Successfully loaded the API key")
except Exception as e:
    print("❌ Error during the load of API Key")
    
    

try:
    print("🚀 Initializing the LLM...")
    llm = ChatGroq(
        model="qwen/qwen3-32b",
        temperature=0,
        max_tokens=None,
        reasoning_format="parsed",
        timeout=None,
        max_retries=2,
        api_key = GROQ_API_KEY
    )
    time.sleep(2)
    print("✅ Successfully Initialized the LLM")
except Exception as e:
    print("❌ Exception during Initializing the LLM")

🚀 Loading API KEY...
✅ Successfully loaded the API key
🚀 Initializing the LLM...
✅ Successfully Initialized the LLM


In [22]:
def read_file(path : str) -> str:
    try:
        print("reading the file ...")
        if not os.path.exists(path):
            raise(ValueError("file donot exists at specified path"))
        with open(path, 'r') as file:
            return file.read()
        print("✅ file read successfully")
    except Exception as e:
        print(f"❌ Error during reading the file")

In [23]:
prompt_file_path = r"prompt_file.txt"
prompt_str = read_file(prompt_file_path)
prompt_template = Template(prompt_str)
question = input("Ask your question: ")
filled_template = prompt_template.substitute(
    QUESTION = question,
    CHUNKS = top_chunks
)
print(filled_template)

reading the file ...
You are HELPFUL RAG ASSISTANT

INPUT

[Question]
Question is a string
what is the pdf about

[Related_chunks]
Related_chunks is a list that contains list of chunk text
['\n\uf0a2 After the scan completes, the N -Stalker Report Manager will prompt \uf0a2 you to \nselect a format for the resulting report as choose Generate HTML. \uf0a2 Review the \nHTML report for vulnerabilities.  \n  \nEXPLORING N-STALKER:  \n  \n• N-Stalker Web Application Security Scanner is a Web security assessment tool.  \n• It incorporates with a well-known', ' the N-Stalker Report Manager will prompt 5. you to \nselect a format for the resulting report as choose Generate HTML.  \n6. Review the HTML report for vulnerabilities.']


Rules to FOLLOW **STRICTLY
 - **ONLY use the ['\n\uf0a2 After the scan completes, the N -Stalker Report Manager will prompt \uf0a2 you to \nselect a format for the resulting report as choose Generate HTML. \uf0a2 Review the \nHTML report for vulnerabilities.  \n  \n

In [24]:
response = llm.invoke(filled_template)
print(response.content)

The PDF is about using the **N-Stalker Web Application Security Scanner**, a tool for assessing web security. It outlines steps such as generating HTML reports after a scan via the N-Stalker Report Manager and reviewing these reports to identify vulnerabilities. The content focuses on the process of conducting security assessments and analyzing results for web applications.
